Performance 🏃 
---
https://github.com/jeffmur/fhe-video-similarity/wiki/experiment

This notebook serves to analyze, compare, and visulize the trade-off of performance when using Fully Homomorphic Encryption (FHE) to compute video similarity scores on mobile vs. desktop devices.

Note: The multi-threading feature for pre-processing is disabled for all experiments, as we value consistency over performance.

Note: Every experiment uses the same encryption scheme & parameters:

* Cryptosystem: CKKS
* Polynomial Degree: 4096
* Encode Scalar: 2^40
* qSizes: [60, 40, 40, 60]

There are three metrics being gather within the application:

⚙️ **Pre-processing Time**: The time it takes to convert the video into a format that can be used for comparison.

📊 **Similarity Scores**: The time it takes to encrypt & compute a similarity score.

# Experiment 1: On-device Comparison

In this experiment, we compare the pre-processing and encryption duration on mobile vs. desktop devices.

We aim to learn how the performance of the application varies amongst devices (mobile vs. desktop). The scenarios are split by resolution (720p, 1080p, 2160p) and the video length is a constant 60 seconds. We select 60 seconds, as there is a direct comparison between the devices, as well as baseline comparison to Pop-Share.

| Device | OS
| --- | --- |
| Samsung S9 | Android
| Pixel 3XL | Android
| PC | Linux
| Raspberry Pi 400 | Linux

In [1]:
from utils.performance_tables import *
from utils import TARGET_SYS, FRAME_COUNTS
# To be summarized in table
kld_err = []
bhatt_err = []
cram_err = []

def add_mean_error(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    kld, bhattacharyya, cramer = mean_error(pathToAssertion, os, frameCounts).values()
    kld_err.append(kld)
    bhatt_err.append(bhattacharyya)
    cram_err.append(cramer)

# Aggregated Pre-processing durations
pp_by_sys = {}
pp_by_res = {}

# Operations FHE vs. Plaintext
ops_by_sys = {}
ops_by_sys_alg = {}
mobile_ops_by_alg = {}

def add_metric(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    # OS pre-processing
    insert_or_append(pp_by_sys, pre_processing_by_sys(pathToAssertion, os, frameCounts))
    
    # Resolution pre-processing
    res = pre_processing_by_res(pathToAssertion.split('/')[1], pathToAssertion, os, frameCounts)
    insert_or_append(pp_by_res, res)
    
    # Mean Error FHE vs. Plaintext
    add_mean_error(pathToAssertion, os, frameCounts)

    # Operations FHE vs. Plaintext
    insert_or_append(ops_by_sys, operations_by_sys(pathToAssertion, os, frameCounts))

    # Operations FHE vs. Plaintext by Algorithm
    insert_or_append(mobile_ops_by_alg, operations_by_alg(pathToAssertion, ['pxl', 's9'], frameCounts))

    # Operations FHE vs. Plaintext by Algorithm & System
    insert_or_append(ops_by_sys_alg, operations_by_sys_alg(pathToAssertion, os, frameCounts))

## Scenario 1: 720p

On every device, import and pre-process the video twice, compare against itself, using two different keys.


### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1280x720 -c:v libx264 -t 60 -an Black_720p_60s.mp4
```

In [2]:
black_720p = "3_performance/720p/60s_Black"
add_metric(black_720p)
verbose_md_table(black_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | 6.23e-11 [100.00%] | 1.25e-10 | 13.50 | 165.00 | 52.00 | 0.57 | 51.43 [9071.08%]
Cramer [pc] [all] | 2.09e-09 [100.00%] | 4.17e-09 | 13.50 | 74.00 | 43.00 | 0.16 | 42.84 [27464.10%]
BC [pc] [all] | 1.00e+00 [100.00%] | 7.78e-10 | 13.50 | 147.00 | 31.00 | 0.13 | 30.87 [23208.27%]
KLD [s9] [all] | -1.14e-11 [100.00%] | 2.29e-11 | 179.50 | 1041.00 | 329.00 | 1.00 | 328.00 [32800.00%]
Cramer [s9] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 179.50 | 531.00 | 306.00 | 0.46 | 305.54 [66712.23%]
BC [s9] [all] | 1.00e+00 [100.00%] | 5.40e-11 | 179.50 | 1062.00 | 195.00 | 0.38 | 194.62 [50947.12%]
KLD [pxl] [all] | -3.75e-12 [100.00%] | 7.51e-12 | 170.50 | 981.00 | 318.00 | 2.00 | 316.00 [15800.00%]
Cramer [pxl] [all] | 1.06e-10 [100.00%] | 2.12e-10 | 170.50 | 486.00 | 292.00 | 0.32 | 291.68 [91723.90%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 4.46e-11 | 170.50 | 970.00 | 183.00 | 0.30 | 182.70 [60296.04%]

### Test 2: Samsung S9

Taken from Handheld Samsung S9, 720p, 60 seconds.

In [3]:
s9_720p = "3_performance/720p/60s_S9"
add_metric(s9_720p)
verbose_md_table(s9_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -2.81e-12 [100.00%] | 5.61e-12 | 74.00 | 159.00 | 47.00 | 0.03 | 46.97 [167757.14%]
Cramer [pc] [all] | 6.54e-10 [100.00%] | 1.31e-09 | 74.00 | 74.00 | 43.00 | 0.02 | 42.98 [268650.00%]
BC [pc] [all] | 1.00e+00 [100.00%] | 5.75e-10 | 74.00 | 154.00 | 30.00 | 0.01 | 29.99 [230669.23%]
KLD [s9] [all] | -5.44e-12 [100.00%] | 1.09e-11 | 411.00 | 1214.00 | 379.00 | 0.07 | 378.93 [541328.57%]
Cramer [s9] [all] | 9.58e-10 [100.00%] | 1.92e-09 | 411.00 | 598.00 | 348.00 | 0.05 | 347.95 [740325.53%]
BC [s9] [all] | 1.00e+00 [100.00%] | 5.00e-11 | 411.00 | 1197.00 | 225.00 | 0.03 | 224.97 [749900.00%]
KLD [pxl] [all] | -2.04e-12 [100.00%] | 4.09e-12 | 452.50 | 1216.00 | 390.00 | 0.06 | 389.94 [660916.95%]
Cramer [pxl] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 452.50 | 605.00 | 366.00 | 0.04 | 365.96 [963057.89%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 1.09e-11 | 452.50 | 1195.00 | 226.00 | 0.03 | 225.97 [807042.86%]

## Scenario 2: 1080p



### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1920x1080 -c:v libx264 -t 60 -an Black_1080p_60s.mp4
```

In [4]:
black_1080p = "3_performance/1080p/60s_Black"
add_metric(black_1080p)
verbose_md_table(black_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | 4.85e-12 [100.00%] | 9.70e-12 | 38.00 | 145.00 | 45.00 | 0.34 | 44.66 [13174.34%]
Cramer [pc] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 38.00 | 71.00 | 41.00 | 0.09 | 40.91 [43517.02%]
BC [pc] [all] | 1.00e+00 [100.00%] | 1.70e-10 | 38.00 | 142.00 | 29.00 | 0.08 | 28.92 [34839.76%]
KLD [s9] [all] | -1.87e-11 [100.00%] | 3.74e-11 | 338.00 | 1192.00 | 384.00 | 1.00 | 383.00 [38300.00%]
Cramer [s9] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 338.00 | 606.00 | 348.00 | 0.67 | 347.33 [52152.25%]
BC [s9] [all] | 1.00e+00 [100.00%] | 1.60e-10 | 338.00 | 1204.00 | 223.00 | 0.49 | 222.51 [45317.52%]
KLD [pxl] [all] | 3.75e-07 [100.00%] | 6.49e-11 | 382.50 | 1234.00 | 407.00 | 1.00 | 406.00 [40600.00%]
Cramer [pxl] [all] | 2.22e-04 [100.00%] | 2.81e-04 | 382.50 | 600.00 | 358.00 | 0.49 | 357.51 [73411.29%]
BC [pxl] [all] | 1.00e+00 [99.98%] | 2.53e-10 | 382.50 | 1197.00 | 223.00 | 0.40 | 222.60 [55510.97%]

### Test 2: Pixel 3XL

Taken from Handheld Pixel 3XL, 1080p, 60 seconds.

In [5]:
pxl_1080p = "3_performance/1080p/60s_PXL"
add_metric(pxl_1080p)
verbose_md_table(pxl_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -6.05e-12 [100.00%] | 1.21e-11 | 327.00 | 145.00 | 45.00 | 0.02 | 44.98 [224900.00%]
Cramer [pc] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 327.00 | 72.00 | 42.00 | 0.02 | 41.98 [262400.00%]
BC [pc] [all] | 1.00e+00 [100.00%] | 3.95e-10 | 327.00 | 145.00 | 29.00 | 0.02 | 28.98 [193233.33%]
KLD [s9] [all] | -1.12e-11 [100.00%] | 2.24e-11 | 814.50 | 1194.00 | 375.00 | 0.07 | 374.93 [559601.49%]
Cramer [s9] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 814.50 | 597.00 | 345.00 | 0.04 | 344.96 [783990.91%]
BC [s9] [all] | 1.00e+00 [100.00%] | 2.39e-11 | 814.50 | 1183.00 | 221.00 | 0.03 | 220.97 [631328.57%]
KLD [pxl] [all] | -2.74e-11 [100.00%] | 5.48e-11 | 1049.50 | 1205.00 | 376.00 | 0.06 | 375.94 [626566.67%]
Cramer [pxl] [all] | 1.10e-09 [100.00%] | 2.21e-09 | 1049.50 | 559.00 | 330.00 | 0.04 | 329.96 [916566.67%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 9.29e-10 | 1049.50 | 1199.00 | 204.00 | 0.03 | 203.97 [728471.43%]

# Experiment 2: FHE vs. Plaintext Operations

In this experiment, we compare the performance of FHE vs. plaintext operations.

## Scenario 1: Absolute Mean Error

Aggregated from the previous experiments, the absolute mean error will be calculated to determine the accuracy of the similarity scores using FHE library.

In [6]:
mean_error_md_table(kld_err, bhatt_err, cram_err)

Function | Mean Error
---|---
KLD | 1.27e-11
Cramer | 1.37e-09
BC | 4.80e-10

## Scenario 2: Android vs. Linux


In [7]:
ops_by_sys

{'pc': {'encryption': [386.0, 387.0, 358.0, 362.0],
  'fhe_compute': [126.0, 120.0, 115.0, 116.0],
  'pt_compute': [0.8560000000000001, 0.057, 0.516, 0.051000000000000004]},
 's9': {'encryption': [2634.0, 3009.0, 3002.0, 2974.0],
  'fhe_compute': [830.0, 952.0, 955.0, 941.0],
  'pt_compute': [1.84, 0.147, 2.157, 0.146]},
 'pxl': {'encryption': [2437.0, 3016.0, 3031.0, 2963.0],
  'fhe_compute': [793.0, 982.0, 988.0, 910.0],
  'pt_compute': [2.621, 0.125, 1.888, 0.12400000000000001]}}

In [8]:
operations_by_sys_md_table(ops_by_sys)

System | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
pc | 373.25 | 119.25 | 0.37
s9 | 2904.75 | 919.50 | 1.07
pxl | 2861.75 | 918.25 | 1.19

In [9]:
ops_by_sys_alg

{'pc': {'kld': {'score': [6.225563178040385e-11,
    -2.8063272173039966e-12,
    4.852151127086222e-12,
    -6.048767635680803e-12],
   'score_perc': [99.99999999377445,
    100.00000000028064,
    99.99999999951478,
    100.00000000060487],
   'err': [1.245112635608077e-10,
    5.612654434607993e-12,
    9.704302254172444e-12,
    1.2097535271361606e-11],
   'pp': [13.5, 74.0, 38.0, 327.0],
   'enc': [165.0, 159.0, 145.0, 145.0],
   'fhe': [52.0, 47.0, 45.0, 45.0],
   'pt': [0.5670000000000001, 0.028, 0.339, 0.02],
   'diff': [51.433, 46.972, 44.661, 44.98],
   'diff_perc': [9071.075837742505,
    167757.14285714287,
    13174.336283185841,
    224900.0]},
  'bhattacharyya': {'score': [0.9999999996110636,
    1.0000000002877316,
    1.0000000000852431,
    1.0000000001976153],
   'score_perc': [99.99999979139773, 99.99999993463624, 100.0, 100.0],
   'err': [7.778727662000051e-10,
    5.754627885323771e-10,
    1.704862917506489e-10,
    3.952306260046612e-10],
   'pp': [13.5, 74.0, 3

In [10]:
operations_by_sys_alg_md_table(ops_by_sys_alg)

System [Algorithm] | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
pc [kld] | 153.50 | 47.25 | 0.24
pc [bhattacharyya] | 147.00 | 29.75 | 0.06
pc [cramer] | 72.75 | 42.25 | 0.07
s9 [kld] | 1160.25 | 366.75 | 0.53
s9 [bhattacharyya] | 1161.50 | 216.00 | 0.23
s9 [cramer] | 583.00 | 336.75 | 0.30
pxl [kld] | 1159.00 | 372.75 | 0.78
pxl [bhattacharyya] | 1140.25 | 209.00 | 0.19
pxl [cramer] | 562.50 | 336.50 | 0.22

In [11]:
mobile_ops_by_alg

{'kld': {'score': [-7.589349734160797e-12,
   -3.742218542974057e-12,
   1.8730600693172554e-07,
   -1.93152545009914e-11],
  'score_perc': [100.00000000075894,
   100.00000000037423,
   99.99998126940632,
   100.00000000193154],
  'err': [1.5178699468321595e-11,
   7.484437085948115e-12,
   5.11517307858357e-11,
   3.86305090019828e-11],
  'pp': [175.0, 431.75, 360.25, 932.0],
  'enc': [1011.0, 1215.0, 1213.0, 1199.5],
  'fhe': [323.5, 384.5, 395.5, 375.5],
  'pt': [1.5, 0.0645, 1.0, 0.0635],
  'diff': [322.0, 384.4355, 394.5, 375.4365],
  'diff_perc': [24300.0, 601122.7602905569, 39450.0, 593084.0796019901]},
 'bhattacharyya': {'score': [0.9999999999976357,
   1.0000000000097897,
   0.9999999530806551,
   1.00000000022622],
  'score_perc': [99.99999999470856,
   99.99999995209102,
   99.98889173619108,
   99.99999994485718],
  'err': [4.9306447813535215e-11,
   3.044703378307645e-11,
   2.066712356807443e-10,
   4.763658911777213e-10],
  'pp': [175.0, 431.75, 360.25, 932.0],
  'enc':

In [12]:
operations_by_alg_md_table(mobile_ops_by_alg)

Algorithm | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
kld | 1159.62 | 369.75 | 0.66
bhattacharyya | 1150.88 | 212.50 | 0.21
cramer | 572.75 | 336.62 | 0.26

# Experiment 4: Pre-processing


## Scenario 1: Android vs. Linux

In [13]:
pp_by_sys

{'pc': [13.5, 74.0, 38.0, 327.0],
 's9': [179.5, 411.0, 338.0, 814.5],
 'pxl': [170.5, 452.5, 382.5, 1049.5]}

In [14]:
pre_processing_by_sys_md_table(pp_by_sys)

System | Average (s) | Min (s) | Max (s)
---|---|---|---
pc | 113.12 | 13.50 | 327.00
s9 | 435.75 | 179.50 | 814.50
pxl | 513.75 | 170.50 | 1049.50

## Scenario 2: Resolution

In [15]:
pp_by_res

{'720p': [121.16666666666667, 312.5],
 '1080p': [252.83333333333334, 730.3333333333334]}

In [16]:
pre_processing_by_res_md_table(pp_by_res)

Resolution | Average (s) | Min (s) | Max (s)
---|---|---|---
720p | 216.83 | 121.17 | 312.50
1080p | 491.58 | 252.83 | 730.33